In [ ]:
# ============================================================
# PATH CONFIGURATION — Adjust for your environment
# ============================================================
# These notebooks were originally developed on Google Colab.
# Paths referencing /content/drive/MyDrive/... are Colab-specific.
#
# For local execution, set ROOT_DIR to your project root:
#
#   import os
#   ROOT_DIR = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
#   # Then replace /content/drive/MyDrive/... paths with os.path.join(ROOT_DIR, ...)
#
# The authoritative dataset files are in: data/train.parquet, data/validation.parquet, data/test.parquet
# Images are in: data/images/
# ============================================================


# Malicious Insurance Document Dataset Generator — V2
## MODULE C — Multimodal Artifact Generation (Images, PDFs, Labels)

**Inherits the finalized Module B (pipeline v2.1, corrected 1:1 design) as
its single source of truth.** Module C performs **zero regeneration** of
anything Module A or Module B produced — it only *reads* `insurance_payload_bank.csv`
and `malicious_text.csv`, and *adds* two new artifact types (`IMAGES/`, `PDF/`)
plus two new label ledgers (`image_labels.csv`, `pdf_labels.csv`) on top.

---

### Design rationale (required documentation)

**Why Module C depends on Module B.**
Module B is the only place in the pipeline that decides *which* payload,
*which* attack family, and *which* `attack_location` belongs to each
malicious sample, under a strict one-document-→-one-malicious-sample
invariant. Re-deciding any of that in Module C would let the image/PDF/text
modalities of the *same* `malicious_id` disagree with each other — which
would silently break the dataset's core claim of being a coherent,
traceable multimodal corpus. Module C therefore treats `malicious_text.csv`
as ground truth for *all* assignment decisions and only renders that
decision into pixels / PDF bytes. No random choice in this module ever
re-decides attack family, payload, or location — those are pure lookups.

**Why Module B outputs must never be modified.**
`malicious_text.csv`, `insurance_payload_bank.csv`, and their manifests have
already been validated and accepted (Module B's own Step 10 validation
report). Any in-place edit from Module C would invalidate that prior
validation without re-running it, and would make `generation_manifest.json`
(Module B's audit trail) describe a file that no longer matches what's on
disk. Module C therefore opens these files in pandas with no write path
back to them anywhere in this notebook.

**Why the one-to-one mapping is preserved.**
Module B already solved the leakage problem: exactly one malicious sample
per benign document, joinable by `source_benign_id` for grouped splitting.
If Module C generated, say, 3 images per malicious text row, the multimodal
dataset would silently reintroduce the leakage risk Module B was written
specifically to eliminate. So Module C generates **exactly one image** and
**at most one PDF** (only if a benign PDF source exists) per row of
`malicious_text.csv`, keyed by the *same* `malicious_id`.

**Why realistic document editing is preferred over synthetic generation.**
A classifier trained on synthetically rendered "fake-looking" insurance
documents learns to detect *rendering artifacts*, not *injected attacks*.
Editing real benign images/PDFs in place — keeping layout, fonts, logos,
tables, and resolution untouched — means the only learnable signal is the
attack itself, which is what makes the dataset useful for an OCR/VLM
pipeline that will see real production documents.

**Why Runtime OCR remains unchanged.**
OCR is intentionally never run during dataset construction (Module A's
frozen design decision, reaffirmed in Module B). Running OCR here would
(a) duplicate work EasyOCR will do at training time, (b) bake a specific
OCR engine's quirks into stored "ground truth," and (c) consume storage on
data that's cheap to recompute at training time. Module C's validation
suite checks file *readability*, not OCR *output*.

**Why batching and multiprocessing improve efficiency.**
Image drawing and PDF page overlay are CPU-bound, per-file, and
**embarrassingly parallel** — file *N* never depends on file *N-1*. Pandas/
NumPy vectorization handles the *metadata* side (joins, validity checks,
distributions) in single-pass array operations, while `ProcessPoolExecutor`
parallelizes the *I/O+rendering* side across all available cores. Using
`executor.map()` (not `as_completed`) keeps result order identical to input
order, so output is deterministic regardless of which worker finishes first
or how many cores are available.

**Why this implementation is suitable for publication-quality multimodal
datasets.** Every artifact is traceable back to `malicious_id` →
`source_benign_id` → `payload_id` → `attack_family` → `attack_location`,
exactly mirroring Module B's schema (no renamed/removed columns — only
additive image/PDF-specific fields appended). Combined with checksum-based
corruption detection, resumability, and a final cross-modality validation
pass, the output is reproducible, leakage-safe, and audit-ready.


In [ ]:
# ============================================================
# MODULE C -- STEP 0: Environment Setup
# ============================================================
import os
import io
import sys
import json
import time
import hashlib
import datetime
import multiprocessing
from concurrent.futures import ProcessPoolExecutor
from functools import lru_cache

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont
from PIL.PngImagePlugin import PngInfo

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, **kwargs):
        return iterable

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except Exception:
    IN_COLAB = False
    print("Not running in Colab -- assuming local/dev environment.")

# GPU availability is only relevant if a GPU-accelerated rendering path
# would beat CPU multiprocessing. PIL drawing and ReportLab/pypdf PDF
# overlay are CPU-bound, I/O-heavy, per-file operations with no batched
# tensor math -- a GPU buys nothing here. We detect it for the summary
# report only and always execute on CPU (see design rationale above).
GPU_AVAILABLE = False
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
except Exception:
    pass

N_CPU = multiprocessing.cpu_count()
COMPUTE_DEVICE = "GPU (detected, unused -- CPU-bound workload)" if GPU_AVAILABLE else "CPU"

_MODULE_C_START_TIME = time.time()

print("Environment ready. IN_COLAB =", IN_COLAB)
print(f"CPU cores available : {N_CPU}")
print(f"GPU available       : {GPU_AVAILABLE}")
print(f"Compute device       : {COMPUTE_DEVICE}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Environment ready. IN_COLAB = True
CPU cores available : 2
GPU available       : False
Compute device       : CPU


In [ ]:
# ============================================================
# MODULE C -- STEP 1: Reconstruct Module B's CONFIG (read-only reuse)
# ============================================================
# This is the EXACT CONFIG dict declared by Module B (pipeline v2.1).
# Module C re-declares it only so it can run standalone after a fresh
# Drive mount -- it never writes to any Module A/B path, and it adds
# no renamed keys, only new additive ones for its own artifacts.

PIPELINE_VERSION_B = "2.1.0"   # the Module B pipeline version this notebook expects
PIPELINE_VERSION_C = "1.0.0"   # Module C's own artifact-generation version
RANDOM_SEED = 42               # inherited -- never re-seeds any Module B decision

DRIVE_ROOT = "/content/drive/MyDrive" if IN_COLAB else os.path.abspath("./drive_sim")

CONFIG = {
    # ---- FROZEN benign source (read-only, never modified) ----
    "BENIGN_ROOT":           os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET"),
    "BENIGN_TEXT_DIR":       os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET", "TEXT_SOURCES"),
    "BENIGN_IMAGE_DIR":
os.path.join(
    DRIVE_ROOT,
    "MASTER_BENIGN_DATASET",
    "TEXT_SOURCES",
    "benign_images"
),
    "BENIGN_TEXT_CSV":       os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET", "TEXT_SOURCES", "benign_text.csv"),
    "BENIGN_IMAGE_META_CSV": os.path.join(DRIVE_ROOT, "MASTER_BENIGN_DATASET", "TEXT_SOURCES", "image_metadata.csv"),

    # ---- malicious project root (single output tree, no duplicates) ----
    "PROJECT_ROOT":   os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET"),
    "PAYLOAD_BANK_DIR": os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "PAYLOAD_BANK"),
    "TEXT_DIR":       os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "TEXT"),
    "IMAGES_DIR":     os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "IMAGES"),
    "PDF_DIR":        os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "PDF"),
    "PARQUET_DIR":    os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "PARQUET"),
    "LABELS_DIR":     os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "LABELS"),
    "VALIDATION_DIR": os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "VALIDATION"),
    "LOGS_DIR":       os.path.join(DRIVE_ROOT, "MASTER_MALICIOUS_DATASET", "LOGS"),

    # ---- design decisions (do not change) ----
    "RUNTIME_OCR_ONLY":    True,   # OCR runs at training time via EasyOCR, never here
    "SINGLE_FINAL_OUTPUT": True,   # one image_labels.csv / pdf_labels.csv, updated in place
    "RESUME_ENABLED":      True,   # every stage checks for existing, valid output first
}

# ---- Module-C-only additive key (new folder, never existed before; created
#      below only if genuinely missing). Not a rename of any existing key. ----
CONFIG.setdefault("BENIGN_PDF_DIR", os.path.join(CONFIG["BENIGN_ROOT"], "PDF_SOURCES"))

# ---- Module B's authoritative outputs (READ-ONLY inputs to Module C) ----
PAYLOAD_BANK_CSV         = os.path.join(CONFIG["PAYLOAD_BANK_DIR"], "insurance_payload_bank.csv")
MALICIOUS_TEXT_CSV       = os.path.join(CONFIG["TEXT_DIR"], "malicious_text.csv")
MALICIOUS_TEXT_META_JSON = os.path.join(CONFIG["TEXT_DIR"], "malicious_text.meta.json")
GENERATION_MANIFEST_JSON = os.path.join(CONFIG["TEXT_DIR"], "generation_manifest.json")

# ---- Module C's own new outputs (additive only, inside existing LABELS_DIR/IMAGES_DIR/PDF_DIR) ----
IMAGE_LABELS_CSV   = os.path.join(CONFIG["LABELS_DIR"], "image_labels.csv")
PDF_LABELS_CSV      = os.path.join(CONFIG["LABELS_DIR"], "pdf_labels.csv")
MODULE_C_MANIFEST_JSON = os.path.join(CONFIG["LABELS_DIR"], "module_c_manifest.json")

print("Module C paths (read-only inputs from Module A/B):")
print("  payload bank   (in) :", PAYLOAD_BANK_CSV)
print("  malicious text (in) :", MALICIOUS_TEXT_CSV)
print("  benign images  (in) :", CONFIG["BENIGN_IMAGE_DIR"])
print("  benign PDFs    (in) :", CONFIG["BENIGN_PDF_DIR"], " (optional, additive)")
print("\nModule C paths (new outputs, additive only):")
print("  malicious images (out):", CONFIG["IMAGES_DIR"])
print("  malicious PDFs   (out):", CONFIG["PDF_DIR"])
print("  image labels     (out):", IMAGE_LABELS_CSV)
print("  pdf labels       (out):", PDF_LABELS_CSV)


def _atomic_save_csv(df, path):
    """Write to .tmp then os.replace -- never leaves a partial file on Drive."""
    tmp = path + ".tmp"
    df.to_csv(tmp, index=False)
    os.replace(tmp, path)


def _atomic_save_json(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "w") as fh:
        json.dump(obj, fh, indent=2, default=str)
    os.replace(tmp, path)


Module C paths (read-only inputs from Module A/B):
  payload bank   (in) : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/PAYLOAD_BANK/insurance_payload_bank.csv
  malicious text (in) : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT/malicious_text.csv
  benign images  (in) : /content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_images
  benign PDFs    (in) : /content/drive/MyDrive/MASTER_BENIGN_DATASET/PDF_SOURCES  (optional, additive)

Module C paths (new outputs, additive only):
  malicious images (out): /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/IMAGES
  malicious PDFs   (out): /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/PDF
  image labels     (out): /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/LABELS/image_labels.csv
  pdf labels       (out): /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/LABELS/pdf_labels.csv


In [ ]:
# ============================================================
# MODULE C -- STEP 2: Folder Reuse + Hard Input Validation
#
# Only creates a folder if it genuinely does not exist (idempotent).
# Never renames, recreates, or duplicates any Module A/B folder.
#
# Hard-stops (raise) if a REQUIRED Module A/B output is missing --
# Module C must never regenerate or repair Module A/B.
# BENIGN_PDF_DIR is a soft requirement: if absent, PDF generation is
# skipped for the whole run (documented, not a hard failure), because
# the brief explicitly makes the PDF artifact conditional on a benign
# PDF actually existing.
# ============================================================
REQUIRED_DIRS = [
    CONFIG["PROJECT_ROOT"], CONFIG["PAYLOAD_BANK_DIR"], CONFIG["TEXT_DIR"],
    CONFIG["IMAGES_DIR"], CONFIG["PDF_DIR"], CONFIG["PARQUET_DIR"],
    CONFIG["LABELS_DIR"], CONFIG["VALIDATION_DIR"], CONFIG["LOGS_DIR"],
]

print("Folder reuse check (existing folders are reused as-is, never renamed):")
for d in REQUIRED_DIRS:
    existed_before = os.path.isdir(d)
    os.makedirs(d, exist_ok=True)
    print(f"  [{'REUSED' if existed_before else 'CREATED (was missing)'}] {d}")

validation_errors = []

if not os.path.exists(PAYLOAD_BANK_CSV):
    validation_errors.append(f"insurance_payload_bank.csv not found at: {PAYLOAD_BANK_CSV} (run Module A)")

if not os.path.exists(MALICIOUS_TEXT_CSV):
    validation_errors.append(f"malicious_text.csv not found at: {MALICIOUS_TEXT_CSV} (run Module B)")

if not os.path.exists(CONFIG["BENIGN_IMAGE_META_CSV"]):
    validation_errors.append(f"image_metadata.csv not found at: {CONFIG['BENIGN_IMAGE_META_CSV']}")

if not os.path.isdir(CONFIG["BENIGN_IMAGE_DIR"]):
    validation_errors.append(f"Benign image source directory not found: {CONFIG['BENIGN_IMAGE_DIR']}")

BENIGN_PDF_DIR_PRESENT = os.path.isdir(CONFIG["BENIGN_PDF_DIR"])
if not BENIGN_PDF_DIR_PRESENT:
    print(f"\nNOTE: benign PDF source directory not found at {CONFIG['BENIGN_PDF_DIR']}.")
    print("      PDF generation will be skipped for this run (images + labels still proceed).")
    print("      This is NOT treated as a hard failure -- the brief makes PDF generation")
    print("      conditional on a benign PDF actually existing.")

if validation_errors:
    print("\n" + "=" * 70)
    print("MODULE C INPUT VALIDATION -- FAILED")
    print("=" * 70)
    for e in validation_errors:
        print("  x", e)
    raise RuntimeError(
        "Module C cannot proceed: required Module A/B output(s) missing. "
        "Module C will NOT regenerate Module A or Module B. Fix the missing "
        "input(s) above, then re-run Module C."
    )
else:
    print("\nAll required Module A/B inputs verified present.")


Folder reuse check (existing folders are reused as-is, never renamed):
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/PAYLOAD_BANK
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/TEXT
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/IMAGES
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/PDF
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/PARQUET
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/LABELS
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/VALIDATION
  [REUSED] /content/drive/MyDrive/MASTER_MALICIOUS_DATASET/LOGS

NOTE: benign PDF source directory not found at /content/drive/MyDrive/MASTER_BENIGN_DATASET/PDF_SOURCES.
      PDF generation will be skipped for this run (images + labels still proceed).
      This is NOT treated as a hard failure -- the brief makes PDF generation
      conditional on a benign PDF actually existing.

All required Modu

In [ ]:
# ============================================================
# MODULE C -- STEP 3: Load Module A/B Outputs (READ-ONLY, never written back)
# ============================================================
payload_bank_df = pd.read_csv(PAYLOAD_BANK_CSV)
REQUIRED_PAYLOAD_COLUMNS = [
    "payload_id", "attack_family", "attack_name", "payload_text",
    "target_component", "severity", "visibility", "ocr_survivability", "description",
]
missing = [c for c in REQUIRED_PAYLOAD_COLUMNS if c not in payload_bank_df.columns]
if missing:
    raise ValueError("insurance_payload_bank.csv is missing required columns: " + str(missing))
if payload_bank_df["payload_id"].duplicated().any():
    raise ValueError("insurance_payload_bank.csv has duplicate payload_id values -- Module B inputs are invalid.")

malicious_text_df = pd.read_csv(MALICIOUS_TEXT_CSV)
MALICIOUS_TEXT_COLUMNS = [
    "malicious_id", "source_benign_id", "source_text_hash", "payload_id",
    "attack_family", "attack_name", "attack_location", "attack_type",
    "visibility", "severity", "ocr_survivability", "target_component",
    "label", "generation_timestamp", "malicious_text",
]
missing = [c for c in MALICIOUS_TEXT_COLUMNS if c not in malicious_text_df.columns]
if missing:
    raise ValueError("malicious_text.csv is missing required columns: " + str(missing) +
                      " -- this indicates Module B output is invalid. Module C will NOT repair it.")
if malicious_text_df["malicious_id"].duplicated().any():
    raise ValueError("malicious_text.csv has duplicate malicious_id values -- Module B output is invalid.")
if malicious_text_df["source_benign_id"].duplicated().any():
    raise ValueError(
        "malicious_text.csv has duplicate source_benign_id values -- the 1:1 invariant "
        "established by Module B has been violated. Module C will NOT proceed or repair this."
    )
if (malicious_text_df["label"] != 1).any():
    raise ValueError("malicious_text.csv contains non-malicious labels -- Module B output is invalid.")

ALLOWED_ATTACK_LOCATIONS = {"header", "footer", "watermark", "metadata", "inline", "tiny", "invisible_unicode"}
bad_loc = malicious_text_df[~malicious_text_df["attack_location"].isin(ALLOWED_ATTACK_LOCATIONS)]
if len(bad_loc) > 0:
    raise ValueError(
        "malicious_text.csv contains attack_location values outside Module B's controlled "
        "vocabulary: " + str(bad_loc["attack_location"].unique().tolist())
    )

N_MALICIOUS = len(malicious_text_df)
print(f"Loaded payload bank   : {len(payload_bank_df):,} payloads, {payload_bank_df['attack_family'].nunique()} families")
print(f"Loaded malicious_text : {N_MALICIOUS:,} rows (1:1 with benign documents, per Module B)")
print(f"Attack locations used : {sorted(malicious_text_df['attack_location'].unique())}")


Loaded payload bank   : 58 payloads, 15 families
Loaded malicious_text : 144,937 rows (1:1 with benign documents, per Module B)
Attack locations used : ['footer', 'header', 'inline', 'invisible_unicode', 'metadata', 'tiny', 'watermark']


In [ ]:
# ============================================================
# MODULE C -- STEP 4: Load Benign Image / PDF Sources (read-only)
# ============================================================
# Reuses the same flexible id/path column detection style as Module B's
# detect_column(), applied here to the image-metadata sidecar.

ID_COLUMN_CANDIDATES = [
    "id", "text_id", "doc_id", "document_id", "benign_id",
    "row_id", "sample_id", "uid", "record_id",
]
PATH_COLUMN_CANDIDATES = [
    "filename", "file_name", "path", "image_path", "file_path",
]

def detect_column(df, candidates, what):
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    for col in df.columns:
        if any(cand in col.lower() for cand in candidates):
            return col
    print(f"WARNING: could not auto-detect a {what} column among {list(df.columns)}")
    return None

benign_image_meta_df = pd.read_csv(CONFIG["BENIGN_IMAGE_META_CSV"])
IMG_ID_COL  = detect_column(benign_image_meta_df, ID_COLUMN_CANDIDATES, "image id")
IMG_PATH_COL = detect_column(benign_image_meta_df, PATH_COLUMN_CANDIDATES, "image path")

if IMG_ID_COL is None or IMG_PATH_COL is None:
    raise ValueError(
        "image_metadata.csv is missing a recognizable id or path column -- "
        f"columns found: {list(benign_image_meta_df.columns)}"
    )

benign_image_meta_df = benign_image_meta_df.rename(
    columns={IMG_ID_COL: "_image_benign_id", IMG_PATH_COL: "_image_filename"}
)[["_image_benign_id", "_image_filename"]].copy()
benign_image_meta_df["_image_benign_id"] = benign_image_meta_df["_image_benign_id"].astype(str)
import os

def repair_image_path(path):

    path = str(path)

    if os.path.exists(path):
        return path

    filename = os.path.basename(path)

    repaired = os.path.join(
        CONFIG["BENIGN_IMAGE_DIR"],
        filename
    )

    if os.path.exists(repaired):
        return repaired

    return None

benign_image_meta_df["_image_full_path"] = (
    benign_image_meta_df["_image_filename"]
    .astype(str)
    .apply(repair_image_path)
)

missing = benign_image_meta_df["_image_full_path"].isna().sum()

if missing:
    raise FileNotFoundError(
        f"{missing} benign image(s) could not be located."
    )

if benign_image_meta_df["_image_benign_id"].duplicated().any():
    raise ValueError("image_metadata.csv has duplicate id values -- benign image source is invalid.")

# ---- Benign PDFs (soft-optional): one PDF file per benign id, matched by
#      filename stem == benign_id, if BENIGN_PDF_DIR exists at all. ----
benign_pdf_lookup = {}
if BENIGN_PDF_DIR_PRESENT:
    for fn in os.listdir(CONFIG["BENIGN_PDF_DIR"]):
        if fn.lower().endswith(".pdf"):
            stem = os.path.splitext(fn)[0]
            benign_pdf_lookup[stem] = os.path.join(CONFIG["BENIGN_PDF_DIR"], fn)

print(f"Benign image sources : {len(benign_image_meta_df):,} rows "
      f"(id col '{IMG_ID_COL}', path col '{IMG_PATH_COL}')")
print(f"Benign PDF sources   : {len(benign_pdf_lookup):,} files found "
      f"({'directory present' if BENIGN_PDF_DIR_PRESENT else 'directory absent -- PDFs skipped'})")


Benign image sources : 3,740 rows (id col 'image_id', path col 'image_path')
Benign PDF sources   : 0 files found (directory absent -- PDFs skipped)


In [ ]:
print(CONFIG["BENIGN_IMAGE_DIR"])

print(
    benign_image_meta_df["_image_full_path"].head(10)
)

print(
    benign_image_meta_df["_image_full_path"].apply(os.path.exists).head(10)
)

/content/drive/MyDrive/MASTER_BENIGN_DATASET/TEXT_SOURCES/benign_images
0    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
1    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
2    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
3    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
4    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
5    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
6    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
7    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
8    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
9    /content/drive/MyDrive/MASTER_BENIGN_DATASET/T...
Name: _image_full_path, dtype: object
0    True
1    True
2    True
3    True
4    True
5    True
6    True
7    True
8    True
9    True
Name: _image_full_path, dtype: bool


In [ ]:
from PIL import Image

img = Image.open(
    benign_image_meta_df["_image_full_path"].iloc[0]
)

print(img.size)

(1654, 2339)


In [ ]:
# ============================================================
# MODULE C -- STEP 5: Vectorised Generation Plan (image + pdf)
#
# Pure pandas merge/map -- no per-row Python, no randomness. Every
# decision (payload, family, location) is already fixed by Module B;
# this step only RESOLVES file paths for that already-fixed decision.
# ============================================================
plan_df = malicious_text_df.merge(
    benign_image_meta_df,
    left_on=malicious_text_df["source_benign_id"].astype(str),
    right_on="_image_benign_id",
    how="left",
)
plan_df = plan_df.drop(columns=[c for c in plan_df.columns if c == "key_0"], errors="ignore")

n_unmatched_images = int(plan_df["_image_full_path"].isna().sum())
if n_unmatched_images > 0:
    print(f"WARNING: {n_unmatched_images:,} malicious_text row(s) have no matching benign image "
          f"(source_benign_id not found in image_metadata.csv). These rows will be skipped for "
          f"image generation but will still be reported in validation as 'missing_image_source'.")

# ---- PDF availability (vectorised dict lookup via .map) ----
plan_df["_pdf_full_path"] = plan_df["source_benign_id"].astype(str).map(benign_pdf_lookup)
n_with_pdf = int(plan_df["_pdf_full_path"].notna().sum())

# ---- Deterministic output filenames (stable across reruns -- pure function of malicious_id) ----
plan_df["_image_output_path"] = plan_df["malicious_id"].apply(
    lambda mid: os.path.join(CONFIG["IMAGES_DIR"], f"{mid}.png")
)
plan_df["_pdf_output_path"] = plan_df["malicious_id"].apply(
    lambda mid: os.path.join(CONFIG["PDF_DIR"], f"{mid}.pdf")
)

print(f"Generation plan built: {len(plan_df):,} rows")
print(f"  Rows with a matched benign image : {len(plan_df) - n_unmatched_images:,}")
print(f"  Rows with a matched benign PDF   : {n_with_pdf:,} "
      f"({'optional artifact, per spec' if n_with_pdf < len(plan_df) else 'all rows'})")
print("\nAttack-location distribution in plan:")
print(plan_df["attack_location"].value_counts().sort_index().to_string())


Generation plan built: 144,937 rows
  Rows with a matched benign image : 3,687
  Rows with a matched benign PDF   : 0 (optional artifact, per spec)

Attack-location distribution in plan:
attack_location
footer               35491
header               35340
inline               25794
invisible_unicode    19325
metadata              9663
tiny                  9662
watermark             9662


In [ ]:
# ============================================================
# MODULE C -- STEP 6: Attack-Location Placement Primitives
#
# attack_location is ALREADY decided by Module B -- these functions only
# RENDER that decision onto an image / PDF page. No randomness, no
# re-assignment. Same 7-value vocabulary as Module B
# (header, footer, watermark, metadata, inline, tiny, invisible_unicode)
# so labels stay perfectly consistent across text/image/pdf modalities.
# ============================================================

@lru_cache(maxsize=4)
def _get_fonts():
    """Cached per-process -- fonts are loaded once per worker, not once per file."""
    try:
        normal = ImageFont.truetype("DejaVuSans.ttf", 16)
        small  = ImageFont.truetype("DejaVuSans.ttf", 11)
        tiny   = ImageFont.truetype("DejaVuSans.ttf", 7)
    except Exception:
        normal = small = tiny = ImageFont.load_default()
    return normal, small, tiny


def _avg_color(img, box):
    crop = img.crop(box)
    arr = np.array(crop.convert("RGB"))
    if arr.size == 0:
        return (255, 255, 255)
    return tuple(int(v) for v in arr.reshape(-1, 3).mean(axis=0))


def render_attack_on_image(img: "Image.Image", payload_text: str, attack_location: str):
    """Transforms an existing legitimate image in place (layout, fonts, logos,
    tables, resolution, background all preserved) by inserting ONLY the
    payload text at the location Module B already assigned.
    Returns the modified PIL Image (RGB)."""
    img = img.convert("RGB")
    w, h = img.size
    normal, small, tiny = _get_fonts()

    if attack_location == "metadata":
        # Not drawn on canvas -- embedded as a PNG text chunk by the caller.
        return img

    draw = ImageDraw.Draw(img)
    # Keep insertions short and bounded so the underlying document layout
    # (tables, logos, spacing) is never visually disturbed.
    short_payload = (payload_text[:140] + "...") if len(payload_text) > 140 else payload_text

    if attack_location == "header":
        draw.text((24, 18), short_payload, fill=(20, 20, 20), font=small)
    elif attack_location == "footer":
        draw.text((24, h - 34), short_payload, fill=(20, 20, 20), font=small)
    elif attack_location == "watermark":
        wm_font = normal
        draw.text((w // 6, h // 2 - 10), short_payload, fill=(190, 190, 190), font=wm_font)
    elif attack_location == "inline":
        # Body region: roughly the vertical middle-left, mimicking an
        # inserted clause within existing paragraph text.
        draw.text((24, h // 2), short_payload, fill=(30, 30, 30), font=small)
    elif attack_location == "tiny":
        box = (24, h - 18, min(w, 320), h - 4)
        draw.text((24, h - 16), short_payload, fill=(60, 60, 60), font=tiny)
    elif attack_location == "invisible_unicode":
        # Pixel analogue of an invisible/zero-width text attack: render in a
        # color matched to the local background so it is not perceptible to
        # a human viewer but is still present in the pixel data (and would
        # survive OCR at training time).
        box = (10, 10, min(w, 280), 28)
        bg = _avg_color(img, box)
        draw.text((10, 12), short_payload, fill=bg, font=small)
    else:
        draw.text((24, 18), short_payload, fill=(20, 20, 20), font=small)

    return img


def save_attacked_image(img, payload_text, attack_location, out_path):
    if attack_location == "metadata":
        info = PngInfo()
        info.add_text("Comment", payload_text[:500])
        img.save(out_path, "PNG", pnginfo=info)
    else:
        img.save(out_path, "PNG")


In [ ]:
# ============================================================
# MODULE C -- STEP 7: Resume / Corruption-Aware Validity Helpers
# ============================================================
def sha256_of_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def is_image_file_valid(path: str) -> bool:
    try:
        if not os.path.exists(path) or os.path.getsize(path) == 0:
            return False
        with Image.open(path) as im:
            im.verify()
        return True
    except Exception:
        return False


def is_pdf_file_valid(path: str) -> bool:
    try:
        if not os.path.exists(path) or os.path.getsize(path) == 0:
            return False
        from pypdf import PdfReader
        reader = PdfReader(path)
        return len(reader.pages) > 0
    except Exception:
        return False


def load_existing_labels(csv_path: str) -> pd.DataFrame:
    if os.path.exists(csv_path):
        try:
            return pd.read_csv(csv_path)
        except Exception:
            return pd.DataFrame()
    return pd.DataFrame()


existing_image_labels_df = load_existing_labels(IMAGE_LABELS_CSV)
existing_pdf_labels_df   = load_existing_labels(PDF_LABELS_CSV)

print(f"Existing image_labels.csv : {len(existing_image_labels_df):,} row(s) on disk")
print(f"Existing pdf_labels.csv   : {len(existing_pdf_labels_df):,} row(s) on disk")


Existing image_labels.csv : 3,687 row(s) on disk
Existing pdf_labels.csv   : 0 row(s) on disk


In [ ]:
# ============================================================
# MODULE C -- STEP 8: Parallel Image Generation (ProcessPoolExecutor)
#
# Each task is fully independent and self-describing (plain dict of
# strings/primitives -- cheap to pickle across process boundaries).
# executor.map() preserves input order in its output, so the result
# list is identical across runs/CPU counts -- required determinism.
# ============================================================
def process_one_image_task(task: dict) -> dict:
    """Runs in a worker process. Pure function of its input dict --
    no shared/global mutable state, no randomness -- fully deterministic."""
    malicious_id   = task["malicious_id"]
    src_path       = task["source_image_path"]
    out_path       = task["output_path"]
    payload_text   = task["payload_text"]
    attack_location = task["attack_location"]

    try:
        with Image.open(src_path) as src_img:
            attacked = render_attack_on_image(src_img, payload_text, attack_location)
            tmp_path = out_path + ".tmp"
            save_attacked_image(attacked, payload_text, attack_location, tmp_path)
            os.replace(tmp_path, out_path)
            w, h = attacked.size
        return {
            "malicious_id": malicious_id,
            "status": "generated",
            "output_path": out_path,
            "image_width": w,
            "image_height": h,
            "file_checksum": sha256_of_file(out_path),
            "error": None,
        }
    except Exception as e:
        return {
            "malicious_id": malicious_id,
            "status": "failed",
            "output_path": out_path,
            "image_width": None,
            "image_height": None,
            "file_checksum": None,
            "error": str(e),
        }


def build_image_tasks(plan_df: pd.DataFrame, existing_labels_df: pd.DataFrame):
    """Vectorised resume decision: an item is skipped only if its output file
    exists, opens cleanly, AND its checksum matches the previously recorded
    checksum (proves the file was not modified/corrupted since last run)."""
    has_image = plan_df["_image_full_path"].notna()
    candidates = plan_df.loc[has_image, [
        "malicious_id", "source_benign_id", "payload_id", "attack_family",
        "attack_name", "attack_location", "attack_type", "visibility",
        "severity", "ocr_survivability", "target_component", "label",
        "_image_full_path", "_image_output_path",
    ]].copy()
    # itertuples() mangles leading-underscore column names into positional
    # fields, so rename to plain names just for the iteration below.
    candidates = candidates.rename(columns={
        "_image_full_path": "image_full_path", "_image_output_path": "image_output_path",
    })

    prev_checksum = {}
    if len(existing_labels_df) > 0 and "file_checksum" in existing_labels_df.columns:
        prev_checksum = dict(zip(existing_labels_df["malicious_id"], existing_labels_df["file_checksum"]))

    tasks, resumed_rows = [], []
    for row in candidates.itertuples(index=False):
        out_path = row.image_output_path
        mid = row.malicious_id
        if is_image_file_valid(out_path):
            current_checksum = sha256_of_file(out_path)
            if prev_checksum.get(mid) == current_checksum or mid not in prev_checksum:
                # File exists, opens fine, and either matches its prior record
                # or there was no prior record (first-time external reuse) --
                # safe to resume without regenerating.
                with Image.open(out_path) as im:
                    w, h = im.size
                resumed_rows.append({
                    "malicious_id": mid, "source_benign_id": row.source_benign_id,
                    "payload_id": row.payload_id, "attack_family": row.attack_family,
                    "attack_name": row.attack_name, "attack_location": row.attack_location,
                    "attack_type": row.attack_type, "visibility": row.visibility,
                    "severity": row.severity, "ocr_survivability": row.ocr_survivability,
                    "target_component": row.target_component, "label": row.label,
                    "status": "resumed", "output_path": out_path,
                    "image_width": w, "image_height": h, "file_checksum": current_checksum,
                    "error": None,
                })
                continue
        # missing, corrupted, or checksum mismatch -> (re)generate
        tasks.append({
            "malicious_id": mid,
            "source_image_path": row.image_full_path,
            "output_path": out_path,
            "payload_text": None,  # filled in by caller from payload_bank_df
            "attack_location": row.attack_location,
            "_meta_row": row,
        })
    return tasks, resumed_rows


In [ ]:
# ============================================================
# MODULE C -- STEP 9: Execute Image Generation (batched, parallel, resumable)
# ============================================================
IMAGE_BATCH_SIZE = 500   # tune for available RAM / Drive throughput

payload_text_by_id = dict(zip(payload_bank_df["payload_id"], payload_bank_df["payload_text"]))

raw_tasks, image_resumed_rows = build_image_tasks(plan_df, existing_image_labels_df)
for t in raw_tasks:
    pid = t["_meta_row"].payload_id
    t["payload_text"] = payload_text_by_id.get(pid, "")
    del t["_meta_row"]

print(f"Image generation plan: {len(image_resumed_rows):,} resumed, {len(raw_tasks):,} to (re)generate, "
      f"running on {N_CPU} core(s) ({COMPUTE_DEVICE}).")

image_generated_rows = []
image_failed_rows = []

if raw_tasks:
    n_workers = max(1, N_CPU)
    for batch_start in range(0, len(raw_tasks), IMAGE_BATCH_SIZE):
        batch = raw_tasks[batch_start: batch_start + IMAGE_BATCH_SIZE]
        with ProcessPoolExecutor(max_workers=n_workers) as executor:
            results = list(executor.map(process_one_image_task, batch))  # order-preserving -> deterministic

        meta_by_id = {t["malicious_id"]: t for t in batch}
        plan_meta_lookup = plan_df.set_index("malicious_id")

        for res in results:
            mid = res["malicious_id"]
            meta = plan_meta_lookup.loc[mid]
            row = {
                "malicious_id": mid, "source_benign_id": meta["source_benign_id"],
                "payload_id": meta["payload_id"], "attack_family": meta["attack_family"],
                "attack_name": meta["attack_name"], "attack_location": meta["attack_location"],
                "attack_type": meta["attack_type"], "visibility": meta["visibility"],
                "severity": meta["severity"], "ocr_survivability": meta["ocr_survivability"],
                "target_component": meta["target_component"], "label": meta["label"],
                "status": res["status"], "output_path": res["output_path"],
                "image_width": res["image_width"], "image_height": res["image_height"],
                "file_checksum": res["file_checksum"], "error": res["error"],
            }
            if res["status"] == "generated":
                image_generated_rows.append(row)
            else:
                image_failed_rows.append(row)

        print(f"  batch {batch_start:,}-{batch_start + len(batch):,}: "
              f"{sum(1 for r in results if r['status'] == 'generated')} generated, "
              f"{sum(1 for r in results if r['status'] == 'failed')} failed")

print(f"\nImage generation complete: {len(image_generated_rows):,} generated, "
      f"{len(image_resumed_rows):,} resumed, {len(image_failed_rows):,} failed.")


Image generation plan: 0 resumed, 3,687 to (re)generate, running on 2 core(s) (CPU).
  batch 0-500: 500 generated, 0 failed
  batch 500-1,000: 500 generated, 0 failed
  batch 1,000-1,500: 500 generated, 0 failed
  batch 1,500-2,000: 500 generated, 0 failed
  batch 2,000-2,500: 500 generated, 0 failed
  batch 2,500-3,000: 500 generated, 0 failed
  batch 3,000-3,500: 500 generated, 0 failed
  batch 3,500-3,687: 187 generated, 0 failed

Image generation complete: 3,687 generated, 0 resumed, 0 failed.


In [ ]:
# ============================================================
# MODULE C -- STEP 10: PDF Generation -- Overlay onto Existing PDFs
#
# Reuses the existing benign PDF byte-for-byte: every original page is
# kept (page count, page order, fonts, layout, tables, embedded images
# all come from pypdf merging the ORIGINAL page with a transparent
# ReportLab overlay containing ONLY the payload text). No PDF is ever
# rebuilt from scratch.
# ============================================================
%pip install reportlab
%pip install pypdf
from reportlab.pdfgen import canvas as _rl_canvas
from pypdf import PdfReader, PdfWriter

def _make_overlay_pdf_bytes(page_width, page_height, payload_text, attack_location):
    buf = io.BytesIO()
    c = _rl_canvas.Canvas(buf, pagesize=(page_width, page_height))
    short_payload = (payload_text[:140] + "...") if len(payload_text) > 140 else payload_text

    if attack_location == "header":
        c.setFont("Helvetica", 9)
        c.setFillColorRGB(0, 0, 0)
        c.drawString(40, page_height - 30, short_payload)
    elif attack_location == "footer":
        c.setFont("Helvetica", 9)
        c.setFillColorRGB(0, 0, 0)
        c.drawString(40, 22, short_payload)
    elif attack_location == "watermark":
        c.saveState()
        c.setFont("Helvetica-Bold", 28)
        from reportlab.lib.colors import Color
        c.setFillColor(Color(0.75, 0.75, 0.75, alpha=0.35))
        c.translate(page_width / 2, page_height / 2)
        c.rotate(35)
        c.drawCentredString(0, 0, short_payload)
        c.restoreState()
    elif attack_location == "inline":
        c.setFont("Helvetica", 8)
        c.setFillColorRGB(0, 0, 0)
        c.drawString(40, page_height / 2, short_payload)
    elif attack_location == "tiny":
        c.setFont("Helvetica", 3.5)
        c.setFillColorRGB(0, 0, 0)
        c.drawString(40, 12, short_payload)
    elif attack_location in ("invisible_unicode", "metadata"):
        # White-on-white text (pixel analogue of an invisible/unicode-hidden
        # attack) -- present in the content stream, invisible to a viewer.
        c.setFont("Helvetica", 6)
        c.setFillColorRGB(1, 1, 1)
        c.drawString(40, page_height - 50, short_payload)
    c.showPage()
    c.save()
    buf.seek(0)
    return buf


def render_attack_on_pdf(src_path: str, out_path: str, payload_text: str, attack_location: str) -> int:
    """Overlays the payload onto the FIRST page of the existing benign PDF,
    preserving every other page untouched. Returns page count."""
    reader = PdfReader(src_path)
    writer = PdfWriter()

    first_page = reader.pages[0]
    pw = float(first_page.mediabox.width)
    ph = float(first_page.mediabox.height)

    if attack_location != "metadata":
        overlay_buf = _make_overlay_pdf_bytes(pw, ph, payload_text, attack_location)
        overlay_reader = PdfReader(overlay_buf)
        first_page.merge_page(overlay_reader.pages[0])

    writer.add_page(first_page)
    for p in reader.pages[1:]:
        writer.add_page(p)

    if attack_location == "metadata":
        meta = dict(reader.metadata) if reader.metadata else {}
        meta["/Subject"] = payload_text[:500]
        writer.add_metadata(meta)

    tmp_path = out_path + ".tmp"
    with open(tmp_path, "wb") as fh:
        writer.write(fh)
    os.replace(tmp_path, out_path)
    return len(writer.pages)


In [ ]:
# ============================================================
# MODULE C -- STEP 11: Execute PDF Generation (batched, parallel, resumable)
# ============================================================
def process_one_pdf_task(task: dict) -> dict:
    malicious_id = task["malicious_id"]
    src_path = task["source_pdf_path"]
    out_path = task["output_path"]
    payload_text = task["payload_text"]
    attack_location = task["attack_location"]
    try:
        page_count = render_attack_on_pdf(src_path, out_path, payload_text, attack_location)
        return {
            "malicious_id": malicious_id, "status": "generated", "output_path": out_path,
            "page_count": page_count, "file_checksum": sha256_of_file(out_path), "error": None,
        }
    except Exception as e:
        return {
            "malicious_id": malicious_id, "status": "failed", "output_path": out_path,
            "page_count": None, "file_checksum": None, "error": str(e),
        }


def build_pdf_tasks(plan_df: pd.DataFrame, existing_labels_df: pd.DataFrame):
    has_pdf = plan_df["_pdf_full_path"].notna()
    candidates = plan_df.loc[has_pdf, [
        "malicious_id", "source_benign_id", "payload_id", "attack_family",
        "attack_name", "attack_location", "attack_type", "visibility",
        "severity", "ocr_survivability", "target_component", "label",
        "_pdf_full_path", "_pdf_output_path",
    ]].copy()
    # itertuples() mangles leading-underscore column names into positional
    # fields, so rename to plain names just for the iteration below.
    candidates = candidates.rename(columns={
        "_pdf_full_path": "pdf_full_path", "_pdf_output_path": "pdf_output_path",
    })

    prev_checksum = {}
    if len(existing_labels_df) > 0 and "file_checksum" in existing_labels_df.columns:
        prev_checksum = dict(zip(existing_labels_df["malicious_id"], existing_labels_df["file_checksum"]))

    tasks, resumed_rows = [], []
    for row in candidates.itertuples(index=False):
        out_path = row.pdf_output_path
        mid = row.malicious_id
        if is_pdf_file_valid(out_path):
            current_checksum = sha256_of_file(out_path)
            if prev_checksum.get(mid) == current_checksum or mid not in prev_checksum:
                from pypdf import PdfReader as _PR
                page_count = len(_PR(out_path).pages)
                resumed_rows.append({
                    "malicious_id": mid, "source_benign_id": row.source_benign_id,
                    "payload_id": row.payload_id, "attack_family": row.attack_family,
                    "attack_name": row.attack_name, "attack_location": row.attack_location,
                    "attack_type": row.attack_type, "visibility": row.visibility,
                    "severity": row.severity, "ocr_survivability": row.ocr_survivability,
                    "target_component": row.target_component, "label": row.label,
                    "status": "resumed", "output_path": out_path,
                    "page_count": page_count, "file_checksum": current_checksum, "error": None,
                })
                continue
        tasks.append({
            "malicious_id": mid, "source_pdf_path": row.pdf_full_path,
            "output_path": out_path, "payload_text": None,
            "attack_location": row.attack_location, "_meta_row": row,
        })
    return tasks, resumed_rows


PDF_BATCH_SIZE = 500

pdf_raw_tasks, pdf_resumed_rows = build_pdf_tasks(plan_df, existing_pdf_labels_df)
for t in pdf_raw_tasks:
    pid = t["_meta_row"].payload_id
    t["payload_text"] = payload_text_by_id.get(pid, "")
    del t["_meta_row"]

print(f"PDF generation plan: {len(pdf_resumed_rows):,} resumed, {len(pdf_raw_tasks):,} to (re)generate, "
      f"running on {N_CPU} core(s) ({COMPUTE_DEVICE}).")

pdf_generated_rows = []
pdf_failed_rows = []

if pdf_raw_tasks:
    n_workers = max(1, N_CPU)
    plan_meta_lookup = plan_df.set_index("malicious_id")
    for batch_start in range(0, len(pdf_raw_tasks), PDF_BATCH_SIZE):
        batch = pdf_raw_tasks[batch_start: batch_start + PDF_BATCH_SIZE]
        with ProcessPoolExecutor(max_workers=n_workers) as executor:
            results = list(executor.map(process_one_pdf_task, batch))  # order-preserving

        for res in results:
            mid = res["malicious_id"]
            meta = plan_meta_lookup.loc[mid]
            row = {
                "malicious_id": mid, "source_benign_id": meta["source_benign_id"],
                "payload_id": meta["payload_id"], "attack_family": meta["attack_family"],
                "attack_name": meta["attack_name"], "attack_location": meta["attack_location"],
                "attack_type": meta["attack_type"], "visibility": meta["visibility"],
                "severity": meta["severity"], "ocr_survivability": meta["ocr_survivability"],
                "target_component": meta["target_component"], "label": meta["label"],
                "status": res["status"], "output_path": res["output_path"],
                "page_count": res["page_count"], "file_checksum": res["file_checksum"],
                "error": res["error"],
            }
            if res["status"] == "generated":
                pdf_generated_rows.append(row)
            else:
                pdf_failed_rows.append(row)

        print(f"  batch {batch_start:,}-{batch_start + len(batch):,}: "
              f"{sum(1 for r in results if r['status'] == 'generated')} generated, "
              f"{sum(1 for r in results if r['status'] == 'failed')} failed")

print(f"\nPDF generation complete: {len(pdf_generated_rows):,} generated, "
      f"{len(pdf_resumed_rows):,} resumed, {len(pdf_failed_rows):,} failed.")


PDF generation plan: 0 resumed, 0 to (re)generate, running on 2 core(s) (CPU).

PDF generation complete: 0 generated, 0 resumed, 0 failed.


In [ ]:
# ============================================================
# MODULE C -- STEP 12: Merge + Atomic In-Place Save (image_labels / pdf_labels)
#
# Backward-compatible schema: every column already present in a prior
# image_labels.csv / pdf_labels.csv is preserved unchanged; only new
# image/PDF-specific columns are appended. Same single file is updated
# in place (no _v2 / _final copies), matching Module A/B's save pattern.
# ============================================================
IMAGE_GEN_TS = datetime.datetime.now(datetime.timezone.utc).isoformat()
PDF_GEN_TS   = datetime.datetime.now(datetime.timezone.utc).isoformat()

new_image_rows_df = pd.DataFrame(image_generated_rows + image_resumed_rows + image_failed_rows)
_prev_img_ts = {}
if len(existing_image_labels_df) > 0 and "image_generation_timestamp" in existing_image_labels_df.columns:
    _prev_img_ts = dict(zip(existing_image_labels_df["malicious_id"], existing_image_labels_df["image_generation_timestamp"]))
if len(new_image_rows_df) > 0:
    new_image_rows_df["image_generation_timestamp"] = new_image_rows_df.apply(
        lambda r: IMAGE_GEN_TS if r["status"] != "resumed" else _prev_img_ts.get(r["malicious_id"], IMAGE_GEN_TS),
        axis=1,
    )

new_pdf_rows_df = pd.DataFrame(pdf_generated_rows + pdf_resumed_rows + pdf_failed_rows)
_prev_pdf_ts = {}
if len(existing_pdf_labels_df) > 0 and "pdf_generation_timestamp" in existing_pdf_labels_df.columns:
    _prev_pdf_ts = dict(zip(existing_pdf_labels_df["malicious_id"], existing_pdf_labels_df["pdf_generation_timestamp"]))
if len(new_pdf_rows_df) > 0:
    new_pdf_rows_df["pdf_generation_timestamp"] = new_pdf_rows_df.apply(
        lambda r: PDF_GEN_TS if r["status"] != "resumed" else _prev_pdf_ts.get(r["malicious_id"], PDF_GEN_TS),
        axis=1,
    )

def merge_labels_in_place(new_df: pd.DataFrame, existing_df: pd.DataFrame, csv_path: str) -> pd.DataFrame:
    if len(new_df) == 0 and len(existing_df) == 0:
        return new_df
    if len(existing_df) > 0:
        combined = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined = new_df
    combined = combined.drop_duplicates(subset=["malicious_id"], keep="last").reset_index(drop=True)
    return combined

image_labels_df = merge_labels_in_place(new_image_rows_df, existing_image_labels_df, IMAGE_LABELS_CSV)
pdf_labels_df    = merge_labels_in_place(new_pdf_rows_df, existing_pdf_labels_df, PDF_LABELS_CSV)

if len(image_labels_df) > 0:
    _atomic_save_csv(image_labels_df, IMAGE_LABELS_CSV)
    image_save_status = "written"
else:
    image_save_status = "nothing_to_save"

if len(pdf_labels_df) > 0:
    _atomic_save_csv(pdf_labels_df, PDF_LABELS_CSV)
    pdf_save_status = "written"
else:
    pdf_save_status = "nothing_to_save"

print(f"image_labels.csv : {len(image_labels_df):,} row(s) -- save status: {image_save_status}")
print(f"pdf_labels.csv   : {len(pdf_labels_df):,} row(s) -- save status: {pdf_save_status}")


image_labels.csv : 3,687 row(s) -- save status: written
pdf_labels.csv   : 0 row(s) -- save status: nothing_to_save


In [ ]:
# ============================================================
# MODULE C -- STEP 13: Full Cross-Modality Validation Suite
# ============================================================
def validate_artifact_labels(df: pd.DataFrame, modality: str, plan_df: pd.DataFrame,
                               malicious_text_df: pd.DataFrame, payload_bank_df: pd.DataFrame) -> dict:
    report = {"errors": [], "warnings": [], "stats": {}}
    if len(df) == 0:
        report["warnings"].append(f"No {modality} labels generated/found.")
        report["is_valid"] = True
        return report

    # ---- duplicate outputs ----
    dup = df[df.duplicated(subset=["malicious_id"], keep=False)]
    if len(dup) > 0:
        report["errors"].append(f"{len(dup):,} duplicate malicious_id row(s) in {modality}_labels.")

    # ---- broken / unreadable files ----
    check_fn = is_image_file_valid if modality == "image" else is_pdf_file_valid
    ok_rows = df[df["status"] != "failed"]
    broken = [p for p in ok_rows["output_path"] if not check_fn(p)]
    if broken:
        report["errors"].append(f"{len(broken):,} {modality} file(s) marked ok but fail to open: {broken[:5]}")

    # ---- explicit failures ----
    n_failed = int((df["status"] == "failed").sum())
    if n_failed > 0:
        report["warnings"].append(f"{n_failed:,} {modality} generation failure(s) recorded.")

    # ---- checksum re-validation (corruption detection) ----
    mismatched = 0
    for _, r in ok_rows.iterrows():
        if os.path.exists(r["output_path"]):
            if sha256_of_file(r["output_path"]) != r["file_checksum"]:
                mismatched += 1
    if mismatched > 0:
        report["errors"].append(f"{mismatched:,} {modality} file(s) have a checksum mismatch (corrupted since recorded).")

    # ---- traceability / metadata synchronization against malicious_text.csv (immutable fields) ----
    immutable_cols = ["source_benign_id", "payload_id", "attack_family", "attack_name", "attack_location", "label"]
    merged_check = df.merge(
        malicious_text_df[["malicious_id"] + immutable_cols],
        on="malicious_id", how="left", suffixes=("", "_truth"),
    )
    drift_rows = 0
    for col in immutable_cols:
        mismatch = merged_check[col].astype(str) != merged_check[f"{col}_truth"].astype(str)
        drift_rows += int(mismatch.sum())
    if drift_rows > 0:
        report["errors"].append(
            f"{drift_rows:,} field-row combination(s) drifted from Module B's malicious_text.csv "
            f"(immutable field changed) -- traceability violated."
        )

    # ---- payload / attack-family consistency against the payload bank ----
    valid_payload_ids = set(payload_bank_df["payload_id"])
    orphan_payloads = df[~df["payload_id"].isin(valid_payload_ids)]
    if len(orphan_payloads) > 0:
        report["warnings"].append(f"{len(orphan_payloads):,} row(s) reference a payload_id outside the current payload bank.")

    # ---- one-to-one mapping: at most one artifact per malicious_id, and every
    #      artifact's malicious_id must exist in malicious_text.csv ----
    valid_mids = set(malicious_text_df["malicious_id"])
    orphan_mids = df[~df["malicious_id"].isin(valid_mids)]
    if len(orphan_mids) > 0:
        report["errors"].append(f"{len(orphan_mids):,} {modality} label row(s) reference a malicious_id not in malicious_text.csv (orphan).")

    if modality == "image":
        expected_mids = set(plan_df.loc[plan_df["_image_full_path"].notna(), "malicious_id"])
        missing_mids = expected_mids - set(df.loc[df["status"] != "failed", "malicious_id"])
        if missing_mids:
            report["errors"].append(
                f"{len(missing_mids):,} malicious_text row(s) with a matched benign image have NO "
                f"corresponding image label -- violates the mandatory 1:1 image requirement."
            )

    report["stats"] = {
        f"total_{modality}_labels": len(df),
        "generated": int((df["status"] == "generated").sum()),
        "resumed": int((df["status"] == "resumed").sum()),
        "failed": int((df["status"] == "failed").sum()),
        "unique_malicious_ids": int(df["malicious_id"].nunique()),
        "attack_family_distribution": df["attack_family"].value_counts().to_dict(),
        "attack_location_distribution": df["attack_location"].value_counts().to_dict(),
    }
    report["is_valid"] = len(report["errors"]) == 0
    return report


image_validation_report = validate_artifact_labels(image_labels_df, "image", plan_df, malicious_text_df, payload_bank_df)
pdf_validation_report    = validate_artifact_labels(pdf_labels_df, "pdf", plan_df, malicious_text_df, payload_bank_df)

print("=" * 70)
print("MODULE C VALIDATION REPORT")
print("=" * 70)
for name, rep in [("IMAGE", image_validation_report), ("PDF", pdf_validation_report)]:
    print(f"\n--- {name} ---")
    print("Status:", "PASSED" if rep["is_valid"] else "FAILED")
    for e in rep["errors"]:
        print("  x", e)
    for w in rep["warnings"]:
        print("  !", w)
    for k, v in rep["stats"].items():
        if isinstance(v, dict):
            print(f"  {k}:")
            for kk, vv in sorted(v.items()):
                print(f"    {kk}: {vv}")
        else:
            print(f"  {k}: {v}")

OVERALL_VALIDATION_PASSED = image_validation_report["is_valid"] and pdf_validation_report["is_valid"]
if not OVERALL_VALIDATION_PASSED:
    print("\nWARNING: validation reported errors above. Review before treating this run as final.")


MODULE C VALIDATION REPORT

--- IMAGE ---
Status: PASSED
  total_image_labels: 3687
  generated: 3687
  resumed: 0
  failed: 0
  unique_malicious_ids: 3687
  attack_family_distribution:
    Context Poisoning: 246
    Data Exfiltration: 246
    Decision Manipulation: 245
    Footer Injection: 246
    Header Injection: 246
    Hidden Multimodal: 245
    Instruction Override: 246
    Insurance-specific Attacks: 246
    Invisible Character Injection: 246
    Metadata Injection: 246
    OCR-specific Attacks: 245
    Policy Bypass: 246
    Tiny Font Injection: 246
    Watermark Injection: 246
    White Text Injection: 246
  attack_location_distribution:
    footer: 909
    header: 893
    inline: 655
    invisible_unicode: 492
    metadata: 246
    tiny: 246
    watermark: 246

--- PDF ---
Status: PASSED
  ! No pdf labels generated/found.


In [ ]:
# ============================================================
# MODULE C -- FINAL SUMMARY
# ============================================================
_elapsed = time.time() - _MODULE_C_START_TIME

total_images_generated = len(image_generated_rows)
total_pdfs_generated    = len(pdf_generated_rows)
total_skipped = len(image_resumed_rows) + len(pdf_resumed_rows)
total_regenerated = total_images_generated + total_pdfs_generated
total_failed = len(image_failed_rows) + len(pdf_failed_rows)

print("=" * 70)
print("MODULE C SUMMARY")
print("=" * 70)
print(f"Project Root        : {CONFIG['PROJECT_ROOT']}")
print(f"Images Generated     : {total_images_generated:,}")
print(f"PDFs Generated       : {total_pdfs_generated:,}")
print(f"Image Labels         : {len(image_labels_df):,}")
print(f"PDF Labels           : {len(pdf_labels_df):,}")
print(f"Skipped (resumed)    : {total_skipped:,}")
print(f"Regenerated          : {total_regenerated:,}")
print(f"Failed               : {total_failed:,}")
print(f"Validation Status    : {'PASSED' if OVERALL_VALIDATION_PASSED else 'FAILED'}")
print(f"Resume Status        : {'enabled' if CONFIG['RESUME_ENABLED'] else 'disabled'}")
print(f"CPU/GPU Used         : {COMPUTE_DEVICE} ({N_CPU} core(s))")
print(f"Processing Time      : {_elapsed:.2f}s")
print("=" * 70)

_atomic_save_json({
    "pipeline_version_c": PIPELINE_VERSION_C,
    "depends_on_pipeline_version_b": PIPELINE_VERSION_B,
    "random_seed": RANDOM_SEED,
    "images_generated": total_images_generated,
    "pdfs_generated": total_pdfs_generated,
    "image_labels_total": len(image_labels_df),
    "pdf_labels_total": len(pdf_labels_df),
    "skipped": total_skipped,
    "regenerated": total_regenerated,
    "failed": total_failed,
    "validation_passed": bool(OVERALL_VALIDATION_PASSED),
    "compute_device": COMPUTE_DEVICE,
    "cpu_cores": N_CPU,
    "processing_time_seconds": round(_elapsed, 2),
    "updated_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "generator": "MODULE_C",
}, MODULE_C_MANIFEST_JSON)

print("MODULE C COMPLETED")


MODULE C SUMMARY
Project Root        : /content/drive/MyDrive/MASTER_MALICIOUS_DATASET
Images Generated     : 3,687
PDFs Generated       : 0
Image Labels         : 3,687
PDF Labels           : 0
Skipped (resumed)    : 0
Regenerated          : 3,687
Failed               : 0
Validation Status    : PASSED
Resume Status        : enabled
CPU/GPU Used         : CPU (2 core(s))
Processing Time      : 3273.24s
MODULE C COMPLETED
